# Preprocessing with TinyShift

This notebook demonstrates the public preprocessing API, its main parameter choices, and safe train/test usage.

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
import sys
import os
sys.path.append(os.path.abspath("../.."))
from tinyshift.preprocessing import (
    FeatureResidualizer,
    RobustGaussianScaler,
    filter_features_by_vif,
)

rng = np.random.default_rng(42)

## Example data

The data contains positive and negative correlation, an independent feature, skewness, and an outlier.

In [3]:
n = 200
base = rng.normal(size=n)
data = pd.DataFrame({
    "base": base,
    "positive_corr": 0.95 * base + rng.normal(scale=0.15, size=n),
    "negative_corr": -0.90 * base + rng.normal(scale=0.20, size=n),
    "independent": rng.normal(size=n),
    "skewed": rng.lognormal(mean=1.0, sigma=0.8, size=n),
})
data.loc[0, "skewed"] = 100.0
data.head()

,base,positive_corr,negative_corr,independent,skewed
0,0.304717,0.340117,-0.310168,0.515410,100.000000
1,-1.039984,-0.776863,0.975341,-0.577539,1.142297
2,0.750451,0.726516,-0.511300,1.274447,3.581150
3,0.940565,0.990127,-0.925256,-0.627588,3.681882
4,-1.951035,-2.161009,1.860165,-0.636615,7.613058


## Variance Inflation Factor filtering

`threshold` controls how aggressively correlated features are removed. `n_jobs=-1` uses all available CPU cores; use `n_jobs=1` for deterministic, low-overhead examples. The result is a Boolean mask aligned with the input columns.

In [4]:
for threshold in (2.5, 5.0, 10.0):
    mask = filter_features_by_vif(data, threshold=threshold, n_jobs=1)
    print(f"threshold={threshold:>4}: {data.columns[mask].tolist()}")

verbose_mask = filter_features_by_vif(
    data, threshold=5.0, verbose=True, n_jobs=1
)

threshold= 2.5: ['positive_corr', 'independent', 'skewed']
threshold= 5.0: ['positive_corr', 'independent', 'skewed']
threshold=10.0: ['positive_corr', 'independent', 'skewed']
Removing feature base with VIF: 47.86
Removing feature negative_corr with VIF: 10.72


## Feature residualization

Residualization keeps every column but replaces selected features with linear-regression residuals. Use `corr_type="abs"` for both positive and negative relationships, or `corr_type="pos"` to target positive relationships only. `corrcoef` must be in `(0, 1]`.

In [5]:
for corr_type in ("abs", "pos"):
    residualizer = FeatureResidualizer(corrcoef=0.8, corr_type=corr_type)
    transformed = residualizer.fit_transform(data)
    transformed = pd.DataFrame(transformed, columns=residualizer.get_feature_names_out())
    print(f"{corr_type=}, residualized columns={list(residualizer.models_)}")
    display(transformed.corr().round(2))

corr_type='abs', residualized columns=[np.int64(0), np.int64(2)]


,base,positive_corr,negative_corr,independent,skewed
base,1.00,-0.00,0.00,0.08,0.06
positive_corr,-0.00,1.00,-0.00,-0.03,0.05
negative_corr,0.00,-0.00,1.00,0.05,-0.04
independent,0.08,-0.03,0.05,1.00,0.06
skewed,0.06,0.05,-0.04,0.06,1.00


corr_type='pos', residualized columns=[np.int64(0)]


,base,positive_corr,negative_corr,independent,skewed
base,1.00,-0.00,-0.18,0.04,0.07
positive_corr,-0.00,1.00,-0.95,-0.03,0.05
negative_corr,-0.18,-0.95,1.00,0.04,-0.06
independent,0.04,-0.03,0.04,1.00,0.06
skewed,0.07,0.05,-0.06,0.06,1.00


## Robust Gaussian scaling

Winsorization accepts the built-in methods `iqr`, `mad`, `stddev`, and `auto`; a quantile specification; fixed bounds; or a callable. Open bounds can be represented by `None`. Yeo-Johnson supports arbitrary finite values, while Box-Cox requires strictly positive input.

In [6]:
interval_methods = {
    "IQR fences": "iqr",
    "MAD interval": "mad",
    "standard deviation": "stddev",
    "automatic selection": "auto",
    "5%-95% quantiles": ("quantile", 0.05, 0.95),
    "upper bound only": (None, 20.0),
    "custom callable": lambda x: (np.quantile(x, 0.02), np.quantile(x, 0.98)),
}

for name, method in interval_methods.items():
    scaler = RobustGaussianScaler(
        winsorize_method=method, power_method="yeo-johnson"
    )
    scaled = scaler.fit_transform(data[["skewed"]])
    print(name, scaler.winsorization_bounds_, scaled.mean().round(6), scaled.std().round(6))

IQR fences [(np.float64(-1.8604479040397663), np.float64(7.607397323293263))] 0.0 1.0
MAD interval [(np.float64(-0.7353991674798408), np.float64(6.308481763074671))] -0.0 1.0
standard deviation [(np.float64(-18.830053922092176), np.float64(27.10113716299694))] 0.0 1.0
automatic selection [(np.float64(-0.7353991674798408), np.float64(6.308481763074671))] -0.0 1.0
5%-95% quantiles [(np.float64(0.6129474193922273), np.float64(10.154482139180725))] -0.0 1.0
upper bound only [(np.float64(-inf), np.float64(20.0))] -0.0 1.0
custom callable [(np.float64(0.4972611723191682), np.float64(12.51201406765051))] 0.0 1.0


In [7]:
positive_data = data[["skewed"]]
for power_method in ("yeo-johnson", "box-cox"):
    scaler = RobustGaussianScaler(
        winsorize_method=("quantile", 0.01, 0.99),
        power_method=power_method,
    )
    scaled = scaler.fit_transform(positive_data)
    print(power_method, scaled.mean().round(6), scaled.std().round(6))

yeo-johnson 0.0 1.0
box-cox 0.0 1.0


## Pipeline and train/test usage

Fit preprocessing only on training data. Calling `transform` on held-out data reuses the learned regressions, clipping bounds, power transformation, and scaling parameters.

In [8]:
X = data.drop(columns="skewed")
y = 2 * base + rng.normal(scale=0.5, size=n)
X_train, X_test = X.iloc[:150], X.iloc[150:]
y_train, y_test = y[:150], y[150:]

pipeline = Pipeline([
    ("residualize", FeatureResidualizer(corrcoef=0.8, corr_type="abs")),
    ("scale", RobustGaussianScaler(winsorize_method="iqr")),
    ("model", LinearRegression()),
])
pipeline.fit(X_train, y_train)
print(f"Train R^2: {pipeline.score(X_train, y_train):.3f}")
print(f"Test R^2:  {pipeline.score(X_test, y_test):.3f}")

Train R^2: 0.938
Test R^2:  0.926
